# Beyond the Missing Pixels — Clean Research Notebook

**A comparative study of classical and learned image inpainting.** This is a reorganized, reproducible *workflow* derived from the original 48-cell exploratory Colab notebook. The original historical notebook remains unchanged on GitHub.

**Start here:** In Colab, select **Runtime → Run all** to load data and the **already trained** U-Net checkpoint and produce evaluations. Training is **opt-in only** below.

**Important:** This is a refactoring of the code, not an exact replay of the historical session. Historical random-mask sequences, optimizer states, random-number draws, and outputs cannot be guaranteed identical. Do not report freshly generated metrics as the original numbers unless independently verified. The official held-out test set should be evaluated only with a fixed model and protocol.

## 1. Environment and Google Drive

In [ ]:
# In Google Colab, packages usually come preinstalled.
# If import fails, uncomment and run:
# %pip install -q torch torchvision opencv-python-headless matplotlib pandas scikit-image
import os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/image_reconstruction')
except ImportError:
    PROJECT_DIR = Path('./image_reconstruction')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('Checkpoint directory:', PROJECT_DIR)

## 2. Dataset and deterministic train/validation split

In [ ]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])
trainval_dataset = datasets.OxfordIIITPet(
    root='./data', split='trainval', target_types='category',
    download=True, transform=transform,
)
train_n = int(0.9 * len(trainval_dataset))
val_n = len(trainval_dataset) - train_n
train_dataset, val_dataset = random_split(
    trainval_dataset, [train_n, val_n],
    generator=torch.Generator().manual_seed(42),
)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
print('Train:', len(train_dataset), 'Validation:', len(val_dataset))

## 3. Mask creation, neural architecture, and masked MAE

In [ ]:
def create_random_masks(images, min_size=20, max_size=60):
    batch_size, channels, height, width = images.shape
    masks = torch.zeros(batch_size, 1, height, width,
                        dtype=images.dtype, device=images.device)
    for i in range(batch_size):
        size = torch.randint(min_size, max_size + 1, (1,)).item()
        x = torch.randint(0, width - size + 1, (1,)).item()
        y = torch.randint(0, height - size + 1, (1,)).item()
        masks[i, :, y:y + size, x:x + size] = 1.0
    return masks

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.layers(x)

class SmallUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = DoubleConv(4, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(128, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = DoubleConv(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = DoubleConv(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = DoubleConv(64, 32)
        self.output = nn.Conv2d(32, 3, 1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return torch.sigmoid(self.output(d1))

def masked_loss(predictions, originals, masks):
    error = torch.abs(predictions - originals) * masks
    return error.sum() / (masks.sum() * originals.shape[1] + 1e-8)

def calculate_mae(original, restored, mask):
    missing = mask.expand_as(original) > 0
    return torch.abs(original - restored)[missing].mean().item()

# Unit test for the previously corrected MAE implementation
_o = torch.zeros(3, 4, 4)
_r = _o.clone()
_m = torch.zeros(1, 4, 4)
_m[:, 1:3, 1:3] = 1
_r[:, 1:3, 1:3] = 0.5
assert abs(calculate_mae(_o, _r, _m) - 0.5) < 1e-6
print('Masked MAE test passed.')
model = SmallUNet()
print('Parameters:', sum(p.numel() for p in model.parameters()))

## 4. Load the existing improved checkpoint (default: do not retrain)

In [ ]:
checkpoint_path = PROJECT_DIR / 'unet_improved.pth'
if not checkpoint_path.exists():
    alternative = PROJECT_DIR / 'models' / 'unet_improved.pth'
    if alternative.exists():
        checkpoint_path = alternative
if not checkpoint_path.exists():
    raise FileNotFoundError(
        'Cannot find unet_improved.pth in Google Drive. '
        'This checkpoint was created in the original Colab session and is NOT committed to GitHub. '
        f'Looked in {PROJECT_DIR} and {PROJECT_DIR / "models"}.'
    )
ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
model.load_state_dict(ckpt['model_state_dict'])
model = model.to('cpu')
model.eval()
print('Loaded:', checkpoint_path)
print('Epochs in improved-training phase:', ckpt.get('epoch'))

## 5. Paired inpainting benchmark implementation

In [ ]:
def evaluate_batch(images, masks, model):
    # IMPORTANT: Use the exact same images and masks for each method.
    images, masks = images.cpu(), masks.cpu()
    model.eval()
    with torch.no_grad():
        damaged = images * (1 - masks)
        predicted = model(torch.cat([damaged, masks], dim=1))
        restored = damaged + predicted * masks
    rows = []
    for i in range(len(images)):
        original, mask = images[i], masks[i]
        result = {'unet_mae': calculate_mae(original, restored[i], mask)}
        original_np = (original.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)
        mask_np = (mask[0].numpy() * 255).astype(np.uint8)
        damaged_np = original_np.copy()
        damaged_np[mask_np > 0] = 0
        damaged_bgr = cv2.cvtColor(damaged_np, cv2.COLOR_RGB2BGR)
        for name, method in [('telea_mae', cv2.INPAINT_TELEA),
                             ('navier_stokes_mae', cv2.INPAINT_NS)]:
            restored_bgr = cv2.inpaint(damaged_bgr, mask_np, 3, method)
            restored_rgb = cv2.cvtColor(restored_bgr, cv2.COLOR_BGR2RGB)
            restored_tensor = torch.from_numpy(restored_rgb.astype(np.float32) / 255).permute(2, 0, 1)
            result[name] = calculate_mae(original, restored_tensor, mask)
        rows.append(result)
    return rows

def evaluate_loader(loader, model, seed, save_file=None):
    # Mask generation is deterministic for this function on an unchanged loader,
    # but the exact historical random sequence is not guaranteed.
    torch.manual_seed(seed)
    rows = []
    for batch_i, (images, _) in enumerate(loader):
        masks = create_random_masks(images)
        batch_results = evaluate_batch(images, masks, model)
        for item in batch_results:
            item['image_id'] = len(rows)
            rows.append(item)
        if save_file is not None and ((batch_i + 1) % 10 == 0):
            pd.DataFrame(rows).to_csv(save_file, index=False)
    df = pd.DataFrame(rows)
    if save_file is not None:
        df.to_csv(save_file, index=False)
    return df

def summarize_results(df):
    cols = {'U-Net': 'unet_mae', 'Telea': 'telea_mae', 'Navier-Stokes': 'navier_stokes_mae'}
    return pd.DataFrame([
        {'method': name, 'mean_masked_mae': df[col].mean(), 'std': df[col].std()}
        for name, col in cols.items()
    ]).sort_values('mean_masked_mae')

## 6. Validation benchmark (368 development images)

This is development data, not an untouched test set. Model selection happened on validation results.

In [ ]:
validation_path = PROJECT_DIR / 'clean_validation_detailed.csv'
validation_df = evaluate_loader(val_loader, model, seed=42, save_file=validation_path)
validation_summary = summarize_results(validation_df)
display(validation_summary)
print('Saved:', validation_path)

## 7. Mask-size sensitivity (100 development images)

Square sizes: 20, 40, 60, 80. **80 px exceeds the trained range.** All methods receive identical masks at each image and size.

In [ ]:
rng = np.random.default_rng(42)
records = []
for image_id in range(min(100, len(val_dataset))):
    original, _ = val_dataset[image_id]
    _, H, W = original.shape
    cx, cy = int(rng.integers(40, W - 40)), int(rng.integers(40, H - 40))
    for size in (20, 40, 60, 80):
        mask = torch.zeros(1, 1, H, W)
        x1, y1 = cx - size // 2, cy - size // 2
        mask[:, :, y1:y1 + size, x1:x1 + size] = 1.0
        measures = evaluate_batch(original.unsqueeze(0), mask, model)[0]
        records.extend({'image_id': image_id, 'mask_size': size,
                        'method': name, 'masked_mae': value}
                       for name, value in [('U-Net', measures['unet_mae']),
                                           ('Telea', measures['telea_mae']),
                                           ('Navier-Stokes', measures['navier_stokes_mae'])])
sweep_df = pd.DataFrame(records)
sweep_df.to_csv(PROJECT_DIR / 'clean_mask_size_detailed.csv', index=False)
sweep_summary = sweep_df.groupby(['mask_size', 'method'])['masked_mae'].agg(['mean', 'std']).reset_index()
display(sweep_summary)
plt.figure(figsize=(8, 5))
for name in ['U-Net', 'Telea', 'Navier-Stokes']:
    part = sweep_summary[sweep_summary['method'] == name]
    plt.plot(part['mask_size'], part['mean'], marker='o', label=name)
plt.xlabel('Square mask width (pixels)')
plt.ylabel('Mean masked MAE (lower is better)')
plt.legend()
plt.tight_layout()
plt.savefig(PROJECT_DIR / 'clean_mask_size_comparison.png', dpi=180)
plt.show()

## 8. Official held-out test benchmark — opt-in

**Do not run repeatedly to tune the model.** The original held-out evaluation is already complete and its reported results are committed in `results/final_test_summary.csv`. The cell below is deliberately disabled by default, to prevent accidental test-set reuse. If doing a fresh verification, enable it only with the model/protocol already fixed.

In [ ]:
RUN_HELD_OUT_TEST = False
if RUN_HELD_OUT_TEST:
    test_dataset = datasets.OxfordIIITPet(
        root='./data', split='test', target_types='category',
        download=True, transform=transform
    )
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)
    final_path = PROJECT_DIR / 'clean_final_test_detailed.csv'
    test_df = evaluate_loader(test_loader, model, seed=2026, save_file=final_path)
    display(summarize_results(test_df))
    print('Saved:', final_path)
else:
    print('Held-out test not rerun. Set RUN_HELD_OUT_TEST=True only for locked-protocol verification.')

## 9. Optional training: checkpoint-safe experimentation (NOT run by default)

The historical baseline was trained for **3 epochs on 500 images** at learning rate 0.001; the improved model received **1 completed additional epoch on 3,312 images** at learning rate 0.0003. This cell is a *new* training procedure for additional work, not an exact replay of the original training session. It preserves historical checkpoints and saves to a new filename.

In [ ]:
RUN_NEW_TRAINING = False
if RUN_NEW_TRAINING:
    train_model = SmallUNet().to(DEVICE)
    train_optimizer = torch.optim.Adam(train_model.parameters(), lr=0.001)
    subset_loader = DataLoader(Subset(train_dataset, range(500)), batch_size=8,
                               shuffle=True, num_workers=2)
    for epoch in range(3):
        train_model.train()
        total = 0.0
        for images, _ in subset_loader:
            images = images.to(DEVICE)
            masks = create_random_masks(images)
            inputs = torch.cat([images * (1 - masks), masks], dim=1)
            loss = masked_loss(train_model(inputs), images, masks)
            train_optimizer.zero_grad()
            loss.backward()
            train_optimizer.step()
            total += loss.item()
        train_path = PROJECT_DIR / 'new_experiment_checkpoint.pth'
        torch.save({'epoch': epoch + 1, 'model_state_dict': train_model.state_dict(),
                    'optimizer_state_dict': train_optimizer.state_dict(),
                    'train_loss': total / len(subset_loader)}, train_path)
        print(f'New training epoch {epoch+1}: {total/len(subset_loader):.4f}; saved {train_path}')
else:
    print('Training skipped — existing improved checkpoint remains unchanged.')

## Known limitations and reproducibility notes

- The historical notebook contains separate exploratory `skimage` exercises, diagnostic cells, repeated training phases, interrupted CPU execution, and checkpoint recovery. These are preserved in the **original** notebook, not executed by default here.
- This cleaned workflow recreates the architecture and metric exactly as observed in the original notebook, while reorganizing the evaluation logic. It is not a claim of byte-identical reproduction of historical results.
- The model checkpoint `unet_improved.pth` is stored on Google Drive, **not in the repository**. You need the actual checkpoint to run model evaluation.
- Original train/validation experiments used random mask placement. Matching a seed does not prove that an old randomly generated mask is paired to the same image if code, random draws, dataset order, or software versions change.
- OpenCV uses 8-bit RGB converted to BGR while U-Net predictions operate on floating-point RGB; this introduces slight quantization differences.
- Model outputs are smooth in complex missing regions; masked MAE alone does not assess perceptual or semantic correctness.
- For future comparisons, save mask coordinates and image IDs explicitly before evaluation, and include environment version locks.